In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import random
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report, confusion_matrix
from fastai.vision.all import *
from PIL import Image  
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as models
import torch.nn.functional as F
import torch.nn as nn
from fastai.learner import Learner
from fastai.data.core import DataLoaders
from fastai.metrics import accuracy
from fastai.callback.all import SaveModelCallback, EarlyStoppingCallback
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
class LiverMRIDataset(Dataset):
    def __init__(self, dataframe, image_column_name, label_column_name, transform=None):
        self.dataframe = dataframe
        self.image_column_name = image_column_name
        self.label_column_name = label_column_name
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe[self.image_column_name].iloc[idx]
        label = self.dataframe[self.label_column_name].iloc[idx]

        # Load the grayscale image
        image = Image.open(img_path).convert('L')  # Convert to grayscale

        # Convert grayscale to 3 channels
        transform_to_3_channel = transforms.Compose([
            transforms.Grayscale(num_output_channels=3)
        ])
        image = transform_to_3_channel(image)

        # Apply other transformations if specified
        if self.transform:
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.long)
        return image, label




class Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.drop_out = nn.Dropout()
        self.linear = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.drop_out(x)
        x = self.linear(x)
        return x


class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        base_model = models.resnet50(pretrained=False)
        encoder_layers = list(base_model.children())
        self.backbone = nn.Sequential(*encoder_layers[:9])  # Use the first 9 layers of ResNet50

    def forward(self, x):
        return self.backbone(x)


class LiverDiseaseModel:
    def __init__(self, train_dataframe_path, test_dataframe_path, image_column_name, label_column_name, split_ratio=0.8, batch_size=8, model_name='resnet_liver_model'):
        self.train_dataframe_path = train_dataframe_path
        self.test_dataframe_path = test_dataframe_path
        self.image_column_name = image_column_name
        self.label_column_name = label_column_name
        self.split_ratio = split_ratio
        self.batch_size = batch_size
        self.model_name = model_name
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load the training and testing datasets
        self.train_df = pd.read_csv(self.train_dataframe_path)
        self.test_df = pd.read_csv(self.test_dataframe_path)

        # Prepare the training and validation datasets
        self.train_transforms = self._get_transforms(train=True)
        self.valid_transforms = self._get_transforms(train=False)

        train_size = int(len(self.train_df) * self.split_ratio)
        valid_size = len(self.train_df) - train_size
        train_subset, valid_subset = random_split(self.train_df, [train_size, valid_size])

        self.train_dataset = LiverMRIDataset(train_subset.dataset, self.image_column_name, self.label_column_name, transform=self.train_transforms)
        self.valid_dataset = LiverMRIDataset(valid_subset.dataset, self.image_column_name, self.label_column_name, transform=self.valid_transforms)
        self.test_dataset = LiverMRIDataset(self.test_df, self.image_column_name, self.label_column_name, transform=self.valid_transforms)

        self.dls = DataLoaders.from_dsets(
            self.train_dataset,
            self.valid_dataset,
            bs=self.batch_size,
            num_workers=4,
            device=self.device
        )

    def _get_transforms(self, train=True):
        if train:
            return transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(15),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            return transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    def _prepare_model(self):
        backbone = Backbone()
        classifier = Classifier(num_classes=2)
        backbone.load_state_dict(torch.load('models/ResNet50.pt'))
        model = nn.Sequential(backbone, classifier)
        return model

    def train(self, base_lr=1e-5, epochs=10):
        model = self._prepare_model()

        learn = Learner(
            self.dls,
            model,
            loss_func=CrossEntropyLossFlat(),
            metrics=[accuracy],
            cbs=[
                SaveModelCallback(fname=self.model_name, monitor='valid_loss'),
                EarlyStoppingCallback(monitor='valid_loss', patience=10)
            ]
        ).to_fp16()

        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs")
            learn.model = torch.nn.DataParallel(learn.model)

        learn.fine_tune(epochs, base_lr=base_lr)

        learn.save(f"{self.model_name}_best_model")
        self.learn = learn  # Save the learner for evaluation

    def evaluate_validation_set(self):
        preds, targs = self.learn.get_preds(dl=self.learn.dls.valid)
        self._print_classification_report(targs, preds)
        self._plot_confusion_matrix(targs, preds)

    def evaluate_test_set(self):
        test_dl = DataLoader(self.test_dataset, batch_size=self.batch_size, num_workers=4)
        dls = DataLoaders(test_dl, device=self.device)
        model = self._prepare_model()
        learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(), metrics=[accuracy]).to_fp16()
        learn.load(f"{self.model_name}_best_model")

        print("\nEvaluating on test data:")
        preds, targs = learn.get_preds(dl=test_dl)
        self._print_classification_report(targs, preds)
        self._plot_confusion_matrix(targs, preds)

    def _print_classification_report(self, targs, preds):
        report = classification_report(targs.cpu(), preds.argmax(dim=1).cpu(), target_names=['No CAD', 'CAD'])
        print(report)

    def _plot_confusion_matrix(self, targs, preds):
        cm = confusion_matrix(targs.cpu(), preds.argmax(dim=1).cpu())
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No CAD', 'CAD'], yticklabels=['No CAD', 'CAD'])
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.title('Confusion Matrix')
        plt.show()

In [ ]:
if __name__ == "__main__":
    model = LiverDiseaseModel(
        train_dataframe_path='Final_Datasets/train_resnet_pancreas.csv',
        test_dataframe_path='Final_Datasets/test_data_incidence.csv',
        image_column_name='FilePath_pancreas',
        label_column_name='CAD',
        split_ratio=0.85,  # Use 80% for training and 20% for validation
        model_name='pancreas_mri_radIM_resnet50_model_nov'
    )

    # Train the model
    model.train(epochs=25, base_lr=1e-5)

    # Evaluate on validation set
    print("\nValidation Set Evaluation:")
    model.evaluate_validation_set()

    # Evaluate on test set
    print("\nTest Set Evaluation:")
    model.evaluate_test_set()

In [ ]:
# class NPYDataset(Dataset):
#     def __init__(self, dataframe, image_column, label_column, first_slice_only=True):
#         self.dataframe = dataframe
#         self.image_column = image_column
#         self.label_column = label_column
#         self.first_slice_only = first_slice_only
#         self.transform = transforms.Compose([
#             transforms.Lambda(lambda x: np.stack([x, x, x], axis=0) if x.ndim == 2 else x),
#             transforms.Lambda(lambda x: x.transpose(1, 2, 0)),
#             transforms.Lambda(lambda x: x.astype(np.float32)),
#             transforms.ToPILImage(),
#             transforms.Resize((224, 224)),
#             transforms.ToTensor(),
#         ])

#     def __len__(self):
#         return len(self.dataframe)

#     def __getitem__(self, idx):
#         npy_path = self.dataframe.iloc[idx][self.image_column]
#         label = self.dataframe.iloc[idx][self.label_column]
        
#         if self.first_slice_only:
#             slice_image = np.load(npy_path)[:, :, :, 0][0]
#         else:
#             raise ValueError("Only first slice mode is currently supported.")

#         slice_image = self.transform(slice_image)
#         label = torch.tensor(label, dtype=torch.long)
        
#         return slice_image, label


# class Classifier(nn.Module):
#     def __init__(self, num_class):
#         super().__init__()
#         self.drop_out = nn.Dropout()
#         self.linear = nn.Linear(2048, num_class)

#     def forward(self, x):
#         x = x.view(x.size(0), -1)
#         x = self.drop_out(x)
#         x = self.linear(x)
#         return x


# class Backbone(nn.Module):
#     def __init__(self):
#         super().__init__()
#         base_model = models.resnet50(pretrained=False)
#         encoder_layers = list(base_model.children())
#         self.backbone = nn.Sequential(*encoder_layers[:9])

#     def forward(self, x):
#         return self.backbone(x)


# class HeartDiseaseModel:
#     def __init__(self, train_dataframe_path, test_dataframe_path=None, image_column='FilePath', label_column='CAD',
#                  batch_size=32, num_class=2, model_name='resnet_medical_model', seed=42):
#         self.train_dataframe_path = train_dataframe_path
#         self.test_dataframe_path = test_dataframe_path
#         self.image_column = image_column
#         self.label_column = label_column
#         self.batch_size = batch_size
#         self.model_name = model_name
#         self.seed = seed
#         self.num_class = num_class
#         self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#         self.best_model_path = f"{self.model_name}_best.pth"
#         self._load_data()

#     def _load_data(self):
#         self.df_train = pd.read_csv(self.train_dataframe_path)
#         if self.test_dataframe_path:
#             self.df_test = pd.read_csv(self.test_dataframe_path)

#     def _get_dataloaders(self, train_df, valid_df):
#         train_dataset = NPYDataset(train_df, image_column=self.image_column, label_column=self.label_column)
#         valid_dataset = NPYDataset(valid_df, image_column=self.image_column, label_column=self.label_column)

#         train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=4)
#         valid_loader = DataLoader(valid_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)
#         return train_loader, valid_loader

#     def _prepare_model(self):
#         # Load the pretrained backbone
#         backbone = Backbone()
#         backbone.load_state_dict(torch.load("/scratch/07880/devansh/CAD/Code_Analysis/models/ResNet50.pt"))  # Load pretrained medical weights
#         classifier = Classifier(num_class=self.num_class)
#         model = nn.Sequential(backbone, classifier)

#         # Use DataParallel for multi-GPU support
#         if torch.cuda.device_count() > 1:
#             print(f"Using {torch.cuda.device_count()} GPUs")
#             model = nn.DataParallel(model)

#         self.model = model.to(self.device)

#         # Optimizer and Loss function
#         self.criterion = nn.CrossEntropyLoss()
#         self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-4)

#     def train_model(self, train_loader, val_loader, num_epochs=10, fold_num=0):
#         min_valid_loss = np.inf

#         for e in range(num_epochs):
#             train_loss = 0.0
#             self.model.train()

#             for data, labels in train_loader:
#                 data, labels = data.to(self.device), labels.to(self.device)
#                 self.optimizer.zero_grad()
#                 output = self.model(data)
#                 loss = self.criterion(output, labels)
#                 loss.backward()
#                 self.optimizer.step()
#                 train_loss += loss.item()

#             # Validation
#             valid_loss = 0.0
#             all_preds, all_labels = [], []
#             self.model.eval()
#             with torch.no_grad():
#                 for data, labels in val_loader:
#                     data, labels = data.to(self.device), labels.to(self.device)
#                     output = self.model(data)
#                     loss = self.criterion(output, labels)
#                     valid_loss += loss.item()
                    
#                     # Collect predictions and labels for accuracy
#                     all_preds.append(output.argmax(dim=1).cpu().numpy())
#                     all_labels.append(labels.cpu().numpy())

#             # Compute validation accuracy
#             all_preds = np.concatenate(all_preds)
#             all_labels = np.concatenate(all_labels)
#             valid_accuracy = accuracy_score(all_labels, all_preds)

#             print(f'Epoch {e+1} | Training Loss: {train_loss/len(train_loader):.4f} | Validation Loss: {valid_loss/len(val_loader):.4f} | Validation Accuracy: {valid_accuracy:.4f}')

#             if min_valid_loss > valid_loss:
#                 print(f'Validation Loss Decreased ({min_valid_loss:.6f} -> {valid_loss:.6f}) | Saving Model')
#                 min_valid_loss = valid_loss
#                 torch.save(self.model.state_dict(), f'acl_fold{fold_num}_best_model.pth')

#     def cross_validate(self, n_splits=5, num_epochs=10):
#         kf = KFold(n_splits=n_splits, shuffle=True, random_state=self.seed)
#         for fold, (train_idx, valid_idx) in enumerate(kf.split(self.df_train)):
#             print(f"\nTraining fold {fold + 1}/{n_splits}")
#             train_df = self.df_train.iloc[train_idx]
#             valid_df = self.df_train.iloc[valid_idx]
#             train_loader, val_loader = self._get_dataloaders(train_df, valid_df)
#             self._prepare_model()
#             self.train_model(train_loader, val_loader, num_epochs=num_epochs, fold_num=fold)

#     def evaluate(self, test_set=False):
#         if test_set and self.test_dataframe_path:
#             self._evaluate_test_data()
#         else:
#             self._evaluate_training_data()

#     def _evaluate_training_data(self):
#         preds, targs = self.model.get_preds(ds_idx=1)  # Validation data evaluation
#         self._print_classification_report(targs, preds)
#         self._plot_confusion_matrix(targs, preds)

#     def _evaluate_test_data(self):
#         test_dataset = NPYDataset(self.df_test, image_column=self.image_column, label_column=self.label_column)
#         test_dl = DataLoader(test_dataset, batch_size=self.batch_size, num_workers=4)
#         preds, targs = self.model.get_preds(dl=test_dl)  # Test data evaluation
#         self._print_classification_report(targs, preds)
#         self._plot_confusion_matrix(targs, preds)

#     def _print_classification_report(self, targs, preds):
#         report = classification_report(targs, preds.argmax(dim=1), target_names=['No CAD', 'CAD'])
#         print(report)

#     def _plot_confusion_matrix(self, targs, preds):
#         cm = confusion_matrix(targs, preds.argmax(dim=1))
#         plt.figure(figsize=(8, 6))
#         sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No CAD', 'CAD'], yticklabels=['No CAD', 'CAD'])
#         plt.xlabel('Predicted')
#         plt.ylabel('True')
#         plt.title('Confusion Matrix')
#         plt.show()

In [ ]:
# if __name__ == "__main__":
#     model = HeartDiseaseModel(
#         train_dataframe_path='Final_Datasets/train_resnet_heart.csv', 
#         test_dataframe_path='Final_Datasets/test_data_incidence.csv', 
#         image_column='FilePath',
#         label_column='CAD',
#         model_name='resnet_medical_model_single_slice',
#         seed=42
#     )
#     model.cross_validate(n_splits=5, num_epochs=10)

#     # Evaluate on the test set with the best model
#     model.evaluate(test_set=True)